# AspectLens: Aspect-Based Sentiment Analysis

**Group 1: Praise, Marya, Ashmit**  
**Course:** PROG74040 - Advanced Topics in Artificial Intelligence and Machine Learning

This notebook establishes the Phase II dataset preparation and TF-IDF Logistic Regression baseline.


## Experiment design

- **Task:** predict sentiment toward a specified aspect in a customer review.
- **Labels:** negative, neutral, positive, and conflict.
- **Dataset:** SemEval restaurant aspect-based sentiment examples from Hugging Face.
- **Baseline:** TF-IDF unigram/bigram features with class-balanced Logistic Regression.
- **Evaluation:** accuracy, macro-F1, weighted-F1, and class-level results on a fixed held-out split.


In [1]:
# Run once in a fresh notebook environment if the dependencies are unavailable.
# %pip install -q datasets "transformers<5" torch accelerate scikit-learn pandas numpy matplotlib seaborn


In [2]:
from pathlib import Path
import inspect
import json
import random
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset, load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
DATASET_ID = "tomaarsen/setfit-absa-semeval-restaurants"
BASE_MODEL = "distilbert-base-uncased"
LABELS = ["negative", "neutral", "positive", "conflict"]
LABEL2ID = {label: index for index, label in enumerate(LABELS)}
ID2LABEL = {index: label for label, index in LABEL2ID.items()}
ROOT = Path.cwd()

random.seed(SEED)
np.random.seed(SEED)


/Users/june/.Trash/assignment-grp-phase2-cleanup-20260811/runtime-artifacts/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load and preprocess the dataset

Preprocessing is deliberately conservative: whitespace is normalized while punctuation, negation, and other sentiment cues are preserved. Duplicate labeled examples and unsupported labels are removed. Each model input explicitly joins the review and target aspect.


In [3]:
def normalize_text(value):
    value = re.sub(r"\s+", " ", str(value or ""))
    return value.strip()


def make_model_input(review, aspect):
    return f"review: {normalize_text(review)} [SEP] aspect: {normalize_text(aspect)}"


def preprocess_split(split):
    frame = split.to_pandas()[["text", "span", "label"]].copy()
    frame["text"] = frame["text"].map(normalize_text)
    frame["span"] = frame["span"].map(normalize_text)
    frame["label"] = frame["label"].map(lambda value: normalize_text(value).lower())
    frame = frame[frame["label"].isin(LABEL2ID)].drop_duplicates().reset_index(drop=True)
    frame["model_input"] = [
        make_model_input(review, aspect)
        for review, aspect in zip(frame["text"], frame["span"])
    ]
    frame["label_id"] = frame["label"].map(LABEL2ID)
    return frame


raw_dataset = load_dataset(DATASET_ID)
prepared = {name: preprocess_split(split) for name, split in raw_dataset.items()}

if "test" in prepared and len(prepared["test"]):
    train_df = prepared["train"].reset_index(drop=True)
    test_df = prepared["test"].reset_index(drop=True)
else:
    train_df, test_df = train_test_split(
        prepared["train"],
        test_size=0.20,
        random_state=SEED,
        stratify=prepared["train"]["label"],
    )
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

print(f"Training examples: {len(train_df):,}")
print(f"Held-out examples: {len(test_df):,}")
train_df[["text", "span", "label", "model_input"]].head()


Training examples: 2,934
Held-out examples: 734


,text,span,label,model_input
0,The menu is limited but almost all of the dish...,dishes,positive,review: The menu is limited but almost all of ...
1,"But, they were too big for the bun.",bun,neutral,"review: But, they were too big for the bun. [S..."
2,"The wait staff is very friendly, if not overly...",wait staff,positive,"review: The wait staff is very friendly, if no..."
3,I recommend the meatballs and caprese salad an...,beans on toast,positive,review: I recommend the meatballs and caprese ...
4,Interesting other dishes for a change include ...,chicken in curry sauc,positive,review: Interesting other dishes for a change ...


In [4]:
label_distribution = pd.concat(
    [
        train_df["label"].value_counts().rename("train"),
        test_df["label"].value_counts().rename("held_out"),
    ],
    axis=1,
).reindex(LABELS).fillna(0).astype(int)
label_distribution


,train,held_out
label,,
negative,635,159
neutral,505,127
positive,1721,430
conflict,73,18


## 2. Train and evaluate the baseline

The baseline is fitted directly below. `class_weight="balanced"` reduces the effect of label imbalance, especially for the rare conflict class.


In [5]:
baseline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_features=30_000,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1_200,
                class_weight="balanced",
                solver="lbfgs",
                random_state=SEED,
            ),
        ),
    ]
)

baseline.fit(train_df["model_input"], train_df["label"])
baseline_predictions = baseline.predict(test_df["model_input"])

baseline_scores = {
    "accuracy": accuracy_score(test_df["label"], baseline_predictions),
    "macro_f1": f1_score(test_df["label"], baseline_predictions, average="macro", zero_division=0),
    "weighted_f1": f1_score(test_df["label"], baseline_predictions, average="weighted", zero_division=0),
}
pd.Series(baseline_scores, name="TF-IDF + Logistic Regression").round(4)


accuracy       0.6649
macro_f1       0.5102
weighted_f1    0.6756
Name: TF-IDF + Logistic Regression, dtype: float64

In [6]:
baseline_report = pd.DataFrame(
    classification_report(
        test_df["label"],
        baseline_predictions,
        labels=LABELS,
        output_dict=True,
        zero_division=0,
    )
).T
baseline_report.loc[LABELS, ["precision", "recall", "f1-score", "support"]].round(3)


,precision,recall,f1-score,support
negative,0.522,0.604,0.560,159.0
neutral,0.436,0.480,0.457,127.0
positive,0.851,0.758,0.802,430.0
conflict,0.185,0.278,0.222,18.0


## Next modeling step

Fine-tune DistilBERT and compare it with this baseline on the same held-out examples.
